# Lab 07: The Vanishing Pink Dots

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-07.ipynb)

**What you will do:** time how long it takes a ring of dots to fade while you fixate, compare your fading time with the book's reported range, and watch a hand-built deep network's backward signal vanish -- and recover with a skip connection.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

The book's own caution for this Lab: "This effect is real, strong, and slightly unsettling...
Stop and blink if it makes you feel dizzy or uneasy, and only try it while seated somewhere
safe -- not while walking or driving."

## The experiment in the book

In Lab 07 you fixated on the centre of a ring of pink dots (or the Lilac Chaser animation)
without moving your eyes, and watched the dots fade -- sometimes replaced by an illusory
rotating green dot -- within seconds, snapping back the moment you blinked or looked away.

## Record your results

Time how long it takes the pink dots to fade while you fixate on the cross (Steps 2-3). Run at
least three trials, blinking and refixating between each one, and note whether you saw the
Lilac Chaser's illusory rotating green dot.

In [ ]:
# example data: replace with your own stopwatch timings, in seconds
troxler_log = pd.DataFrame({
    "trial": [1, 2, 3],
    "seconds_to_fade": [12, 15, 11],       # example data
    "saw_green_dot": [True, True, False],   # example data
})
display(troxler_log)
troxler_log.plot.bar(x="trial", y="seconds_to_fade", legend=False, color="orchid", figsize=(5, 3))
plt.ylabel("seconds to fade"); plt.tight_layout(); plt.show()

## Compare

Lab 07 states that "within 10-20 seconds, the pink dots will begin to fade."

In [ ]:
mean_fade = troxler_log.seconds_to_fade.mean()
print(f"Your mean fade time: {mean_fade:.1f} s")
if mean_fade < 10:
    print("Faster than the book's 10-20 s range -- perhaps your fixation was unusually steady,")
    print("or you started the stopwatch a little late.")
elif mean_fade <= 20:
    print("Inside the book's 10-20 s range.")
else:
    print("Slower than the book's range -- small involuntary eye movements (which refresh the")
    print("image on fresh receptors) can delay fading, so try fixating even more precisely.")

## The AI side

The book compares Troxler fading to the **vanishing gradient problem**: a correction signal
that shrinks as it passes backward through many layers of a deep network, leaving early layers
unable to learn -- until fixes like skip connections (used in ResNet) gave the signal a
shortcut. Below, a tiny hand-built network shows both versions: plain layers, where the
backward signal fades with depth, and the same network with skip connections added.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
rng = np.random.default_rng(2026)
n_layers, width = 20, 16

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_norms(use_skip):
    weights = [rng.normal(0, 0.5, (width, width)) for _ in range(n_layers)]
    x = rng.normal(0, 1, width)

    zs, activations = [], [x]
    for W in weights:
        z = W @ activations[-1]
        a = sigmoid(z)
        if use_skip:
            a = a + activations[-1]   # a shortcut carrying the previous layer's signal forward
        zs.append(z)
        activations.append(a)

    target = rng.normal(0, 1, width)
    grad = activations[-1] - target   # error at the output
    norms = [np.linalg.norm(grad)]
    for W, z in zip(reversed(weights), reversed(zs)):
        d_sigmoid = sigmoid(z) * (1 - sigmoid(z))
        grad_through_layer = W.T @ (grad * d_sigmoid)
        grad = grad_through_layer + grad if use_skip else grad_through_layer
        norms.append(np.linalg.norm(grad))
    return norms[::-1]   # norms[0] = the gradient that reaches the very first layer

norms_plain = gradient_norms(use_skip=False)
norms_skip = gradient_norms(use_skip=True)

plt.figure(figsize=(7, 4))
plt.semilogy(norms_plain, "o-", label="Plain deep network")
plt.semilogy(norms_skip, "s-", label="With skip connections")
plt.xlabel("layer (0 = input side)"); plt.ylabel("gradient norm (log scale)")
plt.title("How far the correction signal reaches back through the network")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Gradient norm reaching layer 0: plain = {norms_plain[0]:.2e}, with skip connections = {norms_skip[0]:.2e}")

The plain network's gradient typically shrinks by several orders of magnitude by the time it
reaches the first layer -- that layer effectively stops learning, the network equivalent of
your neurons no longer bothering to report an unchanging pink dot. The skip-connection version
keeps a healthy gradient at every depth, because the shortcut carries the error signal straight
through, bypassing the repeated multiplication that shrinks it. The disanalogy: your neurons
fade a signal that is genuinely predictable and stays fadeable again as soon as the world
changes, whereas the plain network's vanishing gradient is a fixed structural flaw of its
architecture that never recovers on its own, no matter how much the input changes.

## Questions to think about

1. Your dot-fading trials probably were not identical across the three tries. What differences in how steadily you fixated could account for the differences in your seconds_to_fade?
2. Blinking and moving your eyes brings the dots back instantly. Using the "prediction error" explanation in the Brain Mechanism section, explain why a blink resets the fading process even though it briefly removes all visual input, not just the pink dots.
3. In the gradient-norm plot, the skip-connection network's gradient barely changes with depth, while the plain network's shrinks by several orders of magnitude. Relate this numerically observed pattern to the book's claim that batch normalisation, skip connections, and attention were "developed specifically to prevent important signals from fading as they propagate through the network."
4. The book connects Troxler fading to hedonic adaptation, both good and bad news fading with time. Using the "zero prediction error" explanation of Troxler fading, what would have to be true of an emotional experience for it to resist this kind of fading indefinitely?

## Challenge

Test whether fading depends on the display staying perfectly still on your retina. Try the Lilac
Chaser again, but this time deliberately make small circular movements with your head (keeping
your eyes pointed at the cross) instead of staying completely still. Compare the fading time with
your seated, still-headed trials above.

In [ ]:
# Example data: replace with your own timings.
moving_head_trial = pd.DataFrame({
    "condition": ["head still (from the record cell)", "head moving in small circles"],
    "seconds_to_fade": [12.7, 34.0],   # example data: 12.7 illustrates your mean above
})
display(moving_head_trial)
print("If head movement delays or prevents fading, that supports the book's explanation: small")
print("involuntary eye and head movements normally 'refresh' the image on fresh receptors, and")
print("Troxler fading appears precisely when something -- a very steady fixation, or a stabilised")
print("display -- defeats that natural refresh.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-07.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")